# Phase 3: Rule-based baseline for supplier names and categories

## 1. Get the list of unique suppliers
One row per supplier name (redacted personal-data payments excluded), with number of payments, total spend and the most common expense type.

In [1]:
import re
import duckdb
import pandas as pd
from rapidfuzz import fuzz, process

con = duckdb.connect("../council.duckdb")

suppliers = con.sql("""
    SELECT vendor_raw,
           COUNT(*)    AS n_payments,
           SUM(amount) AS total_spend,
           mode(expense_type) AS main_expense_type
    FROM clean_spend
    WHERE NOT is_redacted
    GROUP BY vendor_raw
    ORDER BY total_spend DESC
""").df()
print(f"Unique supplier names (excluding redacted): {len(suppliers):,}")
suppliers.head(10)

Unique supplier names (excluding redacted): 2,529


,vendor_raw,n_payments,total_spend,main_expense_type
0,LIVERPOOL STREETSCENE SERVICES LIMI,219,23474455.32,Private Contractors (Gen)
1,TRAVEL.CLOUD t/a CLICK TRAVEL LTD,6010,16633750.88,Hotel Bookings-Foreign
2,MATRIX SCM LTD,8304,14753079.24,Agency Staff (Admin/Clerical)
3,MERSEYSIDE WASTE DISPOSAL AUTHORITY,5,14245159.60,Waste Disposal
4,NHS CHESHIRE AND MERSEYSIDE ICB,66,11588416.49,Private Contractors (Gen)
5,TROWERS & HAMLINS LLP,3,10492005.45,Building Acquisitions
6,LIVERPOOL SCHOOLS SERVICES LTD VARI,74,9577962.83,Misc. Expenses - General
7,LIVERPOOL CITY REGION COMBINED AUTH,18,8284770.36,Non Employ Profess Subscript/Membership
8,EDF ENERGY CUSTOMERS LIMITED,52,6515179.18,Electricity (General)
9,AUTISM INITIATIVES,2516,5926716.50,Private Contractors (Gen)


## 2. Rule-based cleaning
Upper-case, `&` → AND, remove punctuation, remove legal words (LTD, LIMITED, PLC…), and drop half-words left by the council's 35-character cut-off.

In [2]:
LEGAL_WORDS = {"LTD", "LIMITED", "PLC", "LLP", "CIC", "INC", "THE", "CO", "COMPANY"}
TRUNCATED_LEN = 35   # the council cuts names off at 35 characters

def clean_name(name):
    s = name.upper()
    s = s.replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)          # punctuation -> space
    words = s.split()
    # if the name was cut off at 35 chars, drop a half-word like "LIMI" or "LIM"
    if len(name) >= TRUNCATED_LEN and words and any(w.startswith(words[-1]) for w in ["LIMITED", "LTD", "SERVICES", "COMPANY"]):
        words = words[:-1]
    words = [w for w in words if w not in LEGAL_WORDS]
    return " ".join(words)

suppliers["clean_key"] = suppliers["vendor_raw"].apply(clean_name)

print("Distinct raw names:        ", suppliers["vendor_raw"].nunique())
print("Distinct after rule clean: ", suppliers["clean_key"].nunique())

# examples where rules merged several spellings
merged = (suppliers.groupby("clean_key")["vendor_raw"]
          .agg(lambda x: " | ".join(sorted(set(x))))
          .loc[lambda s: s.str.contains(r"\|")])
merged.head(15)

Distinct raw names:         2529
Distinct after rule clean:  2476


clean_key
ACORN CARE AND EDUCATION                 ACORN CARE & EDUCATION LTD | Acorn Care and Ed...
ALT VALLEY COMMUNITY TRUST               ALT VALLEY COMMUNITY TRUST | ALT VALLEY COMMUN...
ANFIELD CHILDRENS CENTRE                 ANFIELD CHILDRENS CENTRE | Anfield Childrens C...
ANFIELD SPORTS AND COMMUNITY CENTRE      ANFIELD SPORTS & COMMUNITY CENTRE | Anfield Sp...
ANTHONY WALKER FOUNDATION                ANTHONY WALKER FOUNDATION | The Anthony Walker...
ASYLUM LINK MERSEYSIDE                     ASYLUM LINK MERSEYSIDE | Asylum Link Merseyside
BARRY JONES CONSULTING                   Barry Jones Consulting Limited | Barry Jones C...
BRECKFIELD AND NORTH EVERTON NEIGHBOU    BRECKFIELD & NORTH EVERTON NEIGHBOU | Breckfie...
BRONTE YOUTH AND COMMUNITY CENTRE        BRONTE YOUTH & COMMUNITY CENTRE | Bronte Youth...
CARITAS CARE                                           CARITAS CARE | Caritas Care Limited
CENTRAL YOUTH CLUB                             CENTRAL YOUTH CLUB LTD | Central 

## 3. Fuzzy matching (rapidfuzz)
Merges names that are at least 92% similar, but never if their numbers differ. Check the merged pairs table for wrong merges.

In [3]:
keys = sorted(suppliers["clean_key"].unique())
digits = [re.sub(r"\D", "", k) for k in keys]
THRESHOLD = 92

# similarity of every name against every other name (fast in rapidfuzz)
scores = process.cdist(keys, keys, scorer=fuzz.token_sort_ratio, score_cutoff=THRESHOLD, workers=-1)

# group names that match (union-find)
parent = list(range(len(keys)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

pairs = []
for i in range(len(keys)):
    for j in scores[i].nonzero()[0]:
        if j > i and digits[i] == digits[j]:      # never merge names with different numbers (e.g. L6 vs L8)
            parent[find(j)] = find(i)
            pairs.append((keys[i], keys[j], scores[i][j]))

cluster_of = {k: keys[find(i)] for i, k in enumerate(keys)}
suppliers["cluster"] = suppliers["clean_key"].map(cluster_of)

print("Distinct after fuzzy match:", suppliers["cluster"].nunique())
print(f"Fuzzy pairs merged: {len(pairs)}")
pd.DataFrame(pairs, columns=["name_a", "name_b", "score"]).sort_values("score").head(25)

Distinct after fuzzy match: 2459
Fuzzy pairs merged: 17


,name_a,name_b,score
16,WHITECHAPEL CENTRE,WHITECHAPEL CENTRE SP,92.307693
9,OLD SCHOOL HOUSE DAY NURSERY,OLD SCHOOL HOUSE NURSERY,92.307693
10,PROTECTOR INSURANCE,PROTECTOR INSURANCE UK,92.682930
2,CHILDWALL NURSERY HONEY POT NUR,HONEY POT NURSERY CHILDWALL,93.103447
1,CHILDWALL NURSERY HONEY POT NUR,CHILDWALL NURSERY HONEY POT NURSERY,93.939392
4,GIG MATES,GIGMATES,94.117645
12,SOLACE WOMANS AID,SOLACE WOMENS AID,94.117645
7,HF TRUST,HFT TRUST,94.117645
3,EDUCATION SPECIALISTS,EDUCATIONAL SPECIALISTS,95.454544
0,CHERRY HILL CHILDREN S DAY NURSERY,CHERRY HILL CHILDRENS DAY NURSERY,95.522385


## 4. Rule-based categories and standard names
Keyword rules on the supplier name first, then on the council's expense type. Anything left over is 'Other'. The standard name is the spelling with the most spend in each group.

In [4]:
CATEGORIES = [
    "Social Care", "Education & Childcare", "Construction & Property", "IT & Telecoms",
    "Energy & Utilities", "Transport & Travel", "Waste & Environment", "Professional Services",
    "Staffing & Recruitment", "Housing & Accommodation", "Public Sector Bodies",
    "Community & Voluntary", "Culture, Leisure & Events", "Goods & Supplies", "Other",
]

# keyword rules on the supplier NAME (checked first, in order)
NAME_RULES = [
    ("Public Sector Bodies",     r"\bCOUNCIL\b|\bNHS\b|\bPOLICE\b|\bFIRE\b|\bHMRC\b|PENSION FUND|COMBINED AUTHORITY|\bMBC\b|\bRMBC\b|PUBLIC WORKS LOAN|MERSEYTRAVEL"),
    ("Education & Childcare",    r"SCHOOL|ACADEMY|COLLEGE|UNIVERSITY|NURSERY|CHILDCARE|PRE ?SCHOOL|\bYSGOL\b|LEARNING|EDUCATION|KINDERGARTEN|PLAYGROUP"),
    ("Social Care",              r"\bCARE\b|CARERS|HOMECARE|HOME CARE|NURSING|SUPPORTED LIVING|SUPPORT SERVICES|FOSTER|HEALTHCARE|RESIDENTIAL|DOMICILIARY|\bREST HOME\b|AUTIS|DISABILIT"),
    ("Housing & Accommodation",  r"HOTEL|\bINN\b|HOUSING|HOMES\b|\bLODGE\b|ACCOMMODATION|APARTMENTS|HOSTEL|\bLETTINGS\b|TRAVELODGE|PREMIER INN"),
    ("Staffing & Recruitment",   r"RECRUIT|STAFFING|PERSONNEL|\bAGENCY\b|EMPLOYMENT|LOCUM|\bJOBS\b|TALENT|RESOURCING"),
    ("IT & Telecoms",            r"SOFTWARE|\bIT\b|COMPUTER|DIGITAL|TECHNOLOG|SYSTEMS|TELECOM|COMMUNICATIONS|MICROSOFT|\bDATA\b|NETWORK|\bBT\b|VODAFONE|\bO2\b|CLOUD|CYBER"),
    ("Energy & Utilities",       r"ENERGY|ELECTRIC|\bGAS\b|POWER|WATER|UNITED UTILITIES|EDF|E ON|OCTOPUS|SCOTTISHPOWER|BRITISH GAS|\bFUELS?\b"),
    ("Transport & Travel",       r"COACH|\bBUS\b|TAXI|CARS\b|TRAVEL|TRANSPORT|MOTORS|VEHICLE|\bCABS?\b|\bRAIL\b|AIRPORT|PARKING|LOGISTICS"),
    ("Waste & Environment",      r"WASTE|RECYCL|ENVIRONMENT|CLEANING|STREETSCENE|GROUNDS|LANDSCAP|PEST|TREE|GARDEN|HYGIENE"),
    ("Construction & Property",  r"CONSTRUCT|BUILD|CONTRACTOR|ENGINEER|ROOFING|PLUMB|HEATING|ELECTRICAL|SCAFFOLD|DEMOLITION|CIVIL|SURVEY|ARCHITECT|PROPERTY|ESTATES|JOINERY|GLAZING|PAINTING|DEVELOPMENT|LIFTS?\b|MECH"),
    ("Professional Services",    r"SOLICITOR|\bLAW\b|LEGAL|CHAMBERS|BARRISTER|CONSULT|ACCOUNTANT|AUDIT|ADVISORY|\bKPMG\b|DELOITTE|\bPWC\b|GRANT THORNTON|ASSOCIATES|PARTNERS|INSURANCE|\bBANK\b|FINANCE"),
    ("Community & Voluntary",    r"CHARITY|TRUST\b|FOUNDATION|ASSOCIATION|COMMUNITY|VOLUNTARY|SOCIETY|\bCIO\b|CENTRE|PROJECT|NETWORK|MINISTRIES|CHURCH|MOSQUE|YOUTH|\bCLUB\b|ASYLUM|REFUGEE"),
    ("Culture, Leisure & Events",r"\bARTS?\b|THEATRE|MUSEUM|MUSIC|FESTIVAL|EVENTS|SPORT|LEISURE|CINEMA|GALLERY|FOOTBALL|FILM|MEDIA|CULTURE"),
    ("Goods & Supplies",         r"SUPPLIES|SUPPLY|STATIONERY|FURNITURE|EQUIPMENT|AMAZON|WHOLESALE|TRADING|PRINT|UNIFORM|NISBETS|STORES|DIRECT\b"),
]

# fallback rules on the council's EXPENSE TYPE
EXPENSE_RULES = [
    ("Housing & Accommodation", r"HOTEL|RENT|RSL|ACCOMMODATION"),
    ("Staffing & Recruitment",  r"AGENCY STAFF"),
    ("Education & Childcare",   r"SCH |SCHOOL|EDUC|3 YR|2 YR|NURSERY|ACADEM"),
    ("Transport & Travel",      r"BUS|TRANSPORT|TRAVEL|TAXI|VEHICLE"),
    ("Energy & Utilities",      r"ELECTRIC|GAS|WATER|ENERGY|FUEL"),
    ("Waste & Environment",     r"WASTE|ENVIRONMENT|CLEAN"),
    ("Construction & Property", r"BUILD|STRUCTURES|MAIN CONTRACTOR|HIGHWAYS|R&M|REPAIRS|ACQUISITION"),
    ("IT & Telecoms",           r"SOFTWARE|IT EQUIP|COMPUTER|TELEPHON|COMMUNICATIONS"),
    ("Professional Services",   r"CONSULT|LEGAL|PROFESS|FEES|AUDIT|INSURANCE|BANK"),
    ("Community & Voluntary",   r"GRANT|VOLUNTARY|VOL |CHARIT|PARTNERSHIP"),
    ("Public Sector Bodies",    r"PWLB|PRECEPT|PENSION|OLA|LOANS"),
    ("Culture, Leisure & Events", r"EVENTS|PROMOTION"),
    ("Goods & Supplies",        r"SUPPLIES|EQUIP|FURNITURE|PRINT|STATIONERY|POSTAGE"),
]

def rule_category(name, expense_type):
    n = clean_name(name)
    for cat, pattern in NAME_RULES:
        if re.search(pattern, n):
            return cat
    e = (expense_type or "").upper()
    for cat, pattern in EXPENSE_RULES:
        if re.search(pattern, e):
            return cat
    return "Other"

suppliers["category_rules"] = [rule_category(n, e) for n, e in zip(suppliers["vendor_raw"], suppliers["main_expense_type"])]

# one standard name per cluster: the spelling with the most spend
std = (suppliers.sort_values("total_spend", ascending=False)
       .groupby("cluster")["vendor_raw"].first())
suppliers["std_name_rules"] = suppliers["cluster"].map(std)

summary = (suppliers.groupby("category_rules")
           .agg(suppliers=("vendor_raw", "count"), spend_m=("total_spend", lambda x: round(x.sum() / 1e6, 1)))
           .sort_values("spend_m", ascending=False))
summary

,suppliers,spend_m
category_rules,,
Other,520,62.5
Social Care,207,61.0
Waste & Environment,37,38.6
Education & Childcare,433,37.6
Public Sector Bodies,94,33.2
IT & Telecoms,92,32.9
Community & Voluntary,343,30.8
Construction & Property,142,27.6
Professional Services,253,26.7


## 5. Save the baseline
Saved as a CSV and as a DuckDB table so Phase 4 (LLM) and Phase 5 (evaluation) can use it.

In [5]:
import os
os.makedirs("../data/processed", exist_ok=True)

out = suppliers[["vendor_raw", "std_name_rules", "category_rules", "clean_key",
                 "n_payments", "total_spend", "main_expense_type"]]
out.to_csv("../data/processed/suppliers_rules.csv", index=False)

con.sql("CREATE OR REPLACE TABLE supplier_rules AS SELECT * FROM out")

n_raw = suppliers["vendor_raw"].nunique()
n_rules = suppliers["clean_key"].nunique()
n_fuzzy = suppliers["cluster"].nunique()
other_share = (suppliers["category_rules"] == "Other").mean()

print(f"Supplier names:  {n_raw:,} raw  ->  {n_rules:,} after rules  ->  {n_fuzzy:,} after fuzzy matching")
print(f"Suppliers the rules could not categorise ('Other'): {other_share:.0%}")
print("Saved: data/processed/suppliers_rules.csv and DuckDB table supplier_rules")

Supplier names:  2,529 raw  ->  2,476 after rules  ->  2,459 after fuzzy matching
Suppliers the rules could not categorise ('Other'): 21%
Saved: data/processed/suppliers_rules.csv and DuckDB table supplier_rules
